# Per-channel normalization statistics for 3-channel CXR modes

Compute per-channel `mean`/`std` for each candidate 3-channel representation, so
they can later be plugged into the training `Normalize` step.

**Preprocessing order (deterministic, no augmentation):**

1. read grayscale image (preferred 1024x1024)
2. construct each channel at the **input** resolution (CLAHE / Laplacian / LoG / Sobel / LBP)
3. resize each processed channel to 512x512
4. stack into H x W x 3, values in `[0, 1]`
5. accumulate per-channel sum and squared-sum (streaming, no images kept in memory)

This notebook only **computes and prints** statistics. It does not modify the
training dataloader/config and does not save any files.

In [ ]:
import sys
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Parallelism: use a fraction of visible CPU cores (mirrors scripts/build_mimic_subset.py).
# Change CPU_FRACTION to scale up/down; NUM_WORKERS is derived from it.
# ---------------------------------------------------------------------------
CPU_FRACTION = 0.7
NUM_WORKERS = max(1, int(cpu_count() * CPU_FRACTION))

# OpenCV spins up its own thread pool per call; cap it to 1 so the work is
# parallelised across images (by ThreadPoolExecutor) rather than within a
# single image, which would oversubscribe the CPU.
cv2.setNumThreads(1)

# Resolve repo root (this notebook lives in data/00-examine-data/).
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT:", REPO_ROOT)
print(f"CPU cores: {cpu_count()} | CPU_FRACTION: {CPU_FRACTION} | NUM_WORKERS: {NUM_WORKERS}")

from src.utils.reproducibility import seed_everything
from src.dataloader.utils import resolve_preferred_image_path, resolve_image_path
from src.dataloader.image_channel_preprocessing import (
    build_channels,
    CHANNEL_MODES,
    PreprocessConfig,
)

SEED = 42
seed_everything(SEED)

### Sampling note

The main normalization statistics are computed on **10,000 randomly sampled
training images** (seed 42). We deliberately do **not** oversample long-tail
classes here: normalization should reflect the actual training distribution.

> Tail-enriched sampling can be added later as a diagnostic, but the main
> normalization statistics should come from random training-set sampling.

In [ ]:
TRAIN_DF_PATH = REPO_ROOT / "data" / "data-camchex" / "03_mimic_train.csv"
N_SAMPLE = 10_00
OUT_SIZE = 512

# Modes to characterise.
MODES = [
    "gray3",
    "raw_clahe_sobel",
    "raw_clahe_log",
    "raw_clahe_lbp",
    "raw_clahe_laplacian",
    "raw_clahe_histeq",
    "raw_clahe_clahe",   # raw + mild CLAHE + strong CLAHE (ch2 stats are PROVISIONAL in src; finalize from here)
]

cfg = PreprocessConfig(out_size=OUT_SIZE)
cfg

In [ ]:
# Load the training split only, then randomly sample N_SAMPLE images (seed 42).
df = pd.read_csv(TRAIN_DF_PATH)
if "split" in df.columns:
    df = df[df["split"] == "train"]

n = min(N_SAMPLE, len(df))
sample_df = df.sample(n=n, random_state=SEED).reset_index(drop=True)
paths = sample_df["path"].tolist()
print(f"train rows: {len(df):,} | sampled: {len(paths):,}")

In [ ]:
def read_gray(path):
    """Read a grayscale uint8 image using project path-resolution logic."""
    resolved = resolve_preferred_image_path(path)
    candidates = [resolved]
    if "_resized_1024.jpg" in resolved:
        candidates.append(resolved.replace("_resized_1024.jpg", ".jpg"))
    else:
        candidates.append(resolve_image_path(path))
    for cand in candidates:
        img = cv2.imread(cand, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            return img
    return None


try:
    from tqdm.auto import tqdm
except ImportError:  # pragma: no cover
    def tqdm(x, **kwargs):
        return x

In [ ]:
# Streaming accumulation: never hold processed images in memory.
# Each image is processed in a worker thread (NUM_WORKERS in flight); the
# main thread merges the per-image partial sums as futures complete, so the
# accumulators stay race-free.
acc = {
    m: {
        "sum": np.zeros(3, dtype=np.float64),
        "sumsq": np.zeros(3, dtype=np.float64),
        "pixels": 0,
        "processed": 0,
        "skipped": 0,
    }
    for m in MODES
}

warnings.simplefilter("once")  # quiet the LBP-fallback warning after the first.


def process_one(path):
    """Read one image and return per-mode partial stats (no shared state)."""
    gray = read_gray(path)
    if gray is None:
        return {m: {"skipped": True} for m in MODES}
    result = {}
    for m in MODES:
        try:
            out = build_channels(gray, m, cfg)  # (OUT_SIZE, OUT_SIZE, 3) in [0, 1]
        except Exception:
            result[m] = {"skipped": True}
            continue
        flat = out.reshape(-1, 3).astype(np.float64)
        result[m] = {
            "sum": flat.sum(axis=0),
            "sumsq": (flat ** 2).sum(axis=0),
            "pixels": flat.shape[0],
        }
    return result


with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
    futures = [ex.submit(process_one, path) for path in paths]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="images"):
        result = fut.result()
        for m in MODES:
            r = result[m]
            if r.get("skipped"):
                acc[m]["skipped"] += 1
                continue
            acc[m]["sum"] += r["sum"]
            acc[m]["sumsq"] += r["sumsq"]
            acc[m]["pixels"] += r["pixels"]
            acc[m]["processed"] += 1

print("done")

In [ ]:
# mean = sum / total_pixels ; std = sqrt(sumsq / total_pixels - mean**2)
rows = []
for m in MODES:
    a = acc[m]
    if a["pixels"] > 0:
        mean = a["sum"] / a["pixels"]
        var = a["sumsq"] / a["pixels"] - mean ** 2
        std = np.sqrt(np.clip(var, 0.0, None))
    else:
        mean = np.full(3, np.nan)
        std = np.full(3, np.nan)
    rows.append({
        "mode": m,
        "channels": ",".join(CHANNEL_MODES[m]),
        "mean_ch0": mean[0], "mean_ch1": mean[1], "mean_ch2": mean[2],
        "std_ch0": std[0], "std_ch1": std[1], "std_ch2": std[2],
        "num_processed": a["processed"],
        "num_skipped": a["skipped"],
    })

stats_df = pd.DataFrame(rows, columns=[
    "mode", "channels",
    "mean_ch0", "mean_ch1", "mean_ch2",
    "std_ch0", "std_ch1", "std_ch2",
    "num_processed", "num_skipped",
])

### Results

In [ ]:
from IPython.display import display

display(stats_df)

# Copy-paste-friendly block for the training config.
print()
for m in MODES:
    r = stats_df[stats_df["mode"] == m].iloc[0]
    print(f"{m}:")
    print(f"  mean: [{r['mean_ch0']:.6f}, {r['mean_ch1']:.6f}, {r['mean_ch2']:.6f}]")
    print(f"  std:  [{r['std_ch0']:.6f}, {r['std_ch1']:.6f}, {r['std_ch2']:.6f}]")

### Visual inspection — eyeball every method

For a few randomly-sampled studies, show each channel the modes produce so the
output of every filter can be inspected by eye. Layout per row (one study):

`raw (ch0)` | `clahe (ch1)` | then the distinguishing **channel 2** of each mode
(`sobel`, `log`, `laplacian`, `lbp`).

The edge channels (`sobel`/`log`/`laplacian`) have a per-dataset `std ≈ 0.02`
(see the stats above) — almost all pixels sit near 0, so at their true `[0, 1]`
scale they look nearly black. They are therefore **percentile contrast-stretched
for display only**; the actual values fed to training are the unstretched `[0, 1]`
maps these stats were computed from. `lbp` is dense, so its stretch barely changes it.

In [ ]:
import matplotlib.pyplot as plt

# --- Visual inspection: show every channel method on a handful of studies ----
N_VIS = 5               # number of sample studies to display
VIS_SEED = 7            # resample seed (independent of the SEED used for stats)
STRETCH_PCT = (1, 99)   # display-only contrast stretch for the sparse feature channels

feature_modes = [m for m in MODES if m != "gray3"]  # ch2 differs only for these


def _pstretch(ch, lo_hi=STRETCH_PCT):
    """Percentile contrast-stretch into [0, 1] for *display only* (edge maps are sparse)."""
    lo, hi = np.percentile(ch, lo_hi)
    if hi - lo < 1e-6:
        return np.zeros_like(ch)
    return np.clip((ch - lo) / (hi - lo), 0.0, 1.0)


# Pick N_VIS readable studies (skip any that fail to decode), caching the gray image.
vis = []  # list of (path, gray)
for _p in sample_df["path"].sample(frac=1.0, random_state=VIS_SEED):
    if len(vis) >= N_VIS:
        break
    _g = read_gray(_p)
    if _g is not None:
        vis.append((_p, _g))

col_titles = ["raw\n(ch0)", "clahe\n(ch1)"] + [
    f"{CHANNEL_MODES[m][2]}\n(ch2, stretch)" for m in feature_modes
]
ncols = len(col_titles)
fig, axes = plt.subplots(
    len(vis), ncols, figsize=(2.3 * ncols, 2.5 * len(vis)), squeeze=False
)

for r, (path, gray) in enumerate(vis):
    # raw + clahe are shared across modes -> take them from any clahe stack (true scale).
    base = build_channels(gray, "raw_clahe_sobel", cfg)
    panels = [base[..., 0], base[..., 1]]
    # each mode's distinguishing channel 2 (percentile-stretched for visibility).
    panels += [_pstretch(build_channels(gray, m, cfg)[..., 2]) for m in feature_modes]
    for c, img in enumerate(panels):
        ax = axes[r][c]
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
        if r == 0:
            ax.set_title(col_titles[c], fontsize=9)
    axes[r][0].set_ylabel(Path(str(path)).stem[:24], fontsize=7)

fig.suptitle(
    f"Per-method channel inspection — feature channels (ch2) are "
    f"{STRETCH_PCT[0]}-{STRETCH_PCT[1]}th-pct contrast-stretched for visibility",
    fontsize=10,
)
fig.tight_layout()
plt.show()